In [1]:
# Imports
import sys
from pathlib import Path

# Resolve project root and ensure it's on sys.path
ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "pyproject.toml").exists() or (ROOT / "raw_data").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import discretize_preprocess

In [2]:
# Preprocess data
from pathlib import Path

dataset_path = ROOT / "raw_data" / "adult.csv"
output_path = ROOT / "discretized_data" / "adult.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path), bins=10, strategy='uniform')

Preprocessing: /home/adity/github/katabatic-mentorship-repo/raw_data/adult.csv
Saved preprocessed discrete dataset to: /home/adity/github/katabatic-mentorship-repo/discretized_data/adult.csv


In [3]:
import torch
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.decaf_alex.adapter import KatabaticDECAF

# Device Config
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Set paths
input_csv = str(output_path)
output_dir = str(ROOT / "sample_data" / "adult")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "adult" / "decaf")

# Select protected and target attributes
protected_col = input("Protected Attribute (S): ").strip() or "sex"
target_col = input("Target Attribute (Y): ").strip() or "class"

# Construct Causal DAG with edges [Parent -> Child]
# DECAF uses this to mask the generator weights.
adult_dag = [
    ['age', 'marital-status'],
    ['sex', 'marital-status'],
    ['sex', 'education'],
    ['race', 'education'],
    ['education', 'occupation'],
    ['education', 'hours-per-week'],
    ['marital-status', 'relationship'],
    ['occupation', 'class'],
    ['hours-per-week', 'class'],
    # edge for [protected attribute -> target] (to be debiased later)
    [protected_col, target_col] 
]

# DECAF parameters
model_config = {
    "epochs": 50,
    "batch_size": 64,
    "dag": adult_dag,

    "fairness_config": {
        "S": protected_col,
        "Y": target_col,
        "S_under": "0", #underrepresented class
        "Y_desire": "1" #class to align to
    }
}

pipeline = TrainTestSplitPipeline(model=KatabaticDECAF)

pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
    **model_config
)

/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Loading DECAF training data from: /home/adity/github/katabatic-mentorship-repo/sample_data/adult
Initializing DECAF (Dims:15, DAG Edges:10) on cuda...
Training DECAF on 26048 rows for 50 epochs...


Training DECAF:   0%|          | 0/50 [00:00<?, ?it/s]/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Training DECAF: 100%|██████████| 50/50 [05:41<00:00,  6.84s/it, d_loss=-0.714, g_loss=4.97]


Generating DECAF synthetic data to: /home/adity/github/katabatic-mentorship-repo/synthetic/adult/decaf
Saved split artifacts: x_synth ((1001, 14)), y_synth ((1001,))

Results saved to: Results/adult/decaf_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.4969
F1 Score: 0.5165
AUC: 0.7686

MLP:
Accuracy: 0.4952
F1 Score: 0.5154
AUC: 0.5500

RF:
Accuracy: 0.7895
F1 Score: 0.7650
AUC: 0.7411

XGBoost:
Accuracy: 0.5469
F1 Score: 0.5775
AUC: 0.5008


'Train test split pipeline executed successfully.'